## Step 1: The Guest List (a.k.a. File Name Collector)

Before we can stare at distant stars for hours, we need to know *which* stars we're stalking. This script builds a CSV registry of Kepler `.fits.gz` file names we intend to process later — think of it as the master checklist for the entire pipeline that follows in Notebooks 2–4.

**Why this exists:** Kepler Target Pixel Files(`TPFs`) have long, cryptic names like `kplr005780885-2009131105131_lpd-targ.fits.gz`. Typing these manually every time you run a script would be miserable, so instead we type them *once*, save them to a CSV, and every downstream notebook just reads from that file.

**The Algorithm:**
1. **Check for an existing guest list** — if `names.fitz.gz.csv` already exists, ask the user: *"add more names, or start fresh?"*  This prevents accidentally nuking a list you spent 20 minutes typing.
2. **Collect file names** one at a time via `input()`, until the user types `0`. No validation happens here — it trusts you completely, which is either sweet or terrifying depending on your typing accuracy.
3. **Deduplicate** the list with `pandas.DataFrame.drop_duplicates(subset='name')` — because nobody needs the same star twice, no matter how pretty it is. This matters a lot if you're running this cell across multiple sessions and might accidentally re-add a file.
4. **Append to CSV** with `mode='a'` — new names get bolted onto the existing file rather than overwriting it, Pandas doing the heavy lifting of I/O so you don't have to `open()`/`write()`/`close()` like a caveman.
5. **Print the final roster** (and `.shape`) so you can confirm how many files are currently tracked and admire your collection of cryptic filenames like a proud stellar hoarder.

**Output:** `names.fitz.gz.csv` — a single-column CSV (`name`) that every later notebook (2, 3, 4) reads as its starting point.

**Gotcha to know:** because this uses `mode='a'` on every run, if you run the cell twice with the *same* fresh list, you can get duplicate rows in the CSV itself (dedup only happens *within* a single run's `dc`, not against the file on disk). Worth a `drop_duplicates()` pass on the saved CSV occasionally.

In [1]:
import pandas as pd
import os

csv_filename = "names.fitz.gz.csv"

if os.path.exists(csv_filename):
    choice = input(
        "The file already exists.\n"
        "Enter 'A' or 'a' to add more file names or 'S' or 's' to start from the beginning: "
    ).strip().upper()

    if choice == "A" or choice == 'a':
        df = pd.read_csv(csv_filename)
        file_names = df["name"].tolist()
    else:
        file_names = []
else:
    file_names = []

while True:
    name = input("Enter a file name (or 0 to finish): ")

    if name == "0":
        break

    file_names.append(name)

data = {"name": file_names}
df = pd.DataFrame(data)

dc = df.drop_duplicates(subset='name')

dc.to_csv(csv_filename,mode='a', index=False)

print("\nDataFrame:")
print(dc)
print(f"\nCSV file '{csv_filename}' has been created/updated successfully.")

The file already exists.
Enter 'A' or 'a' to add more file names or 'S' or 's' to start from the beginning:  s
Enter a file name (or 0 to finish):  kplr005780885-2009131105131_lpd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2009166043257_lpd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2009291181958_spd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2010326094124_spd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2011024051157_spd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2011053090032_spd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2011073133259_spd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr005780885-2011177032512_lpd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr008191672-2010140023957_spd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr009651668-2009166043257_lpd-targ.fits.gz
Enter a file name (or 0 to finish):  kplr009651668-2011271113734_lpd-


DataFrame:
                                            name
0   kplr005780885-2009131105131_lpd-targ.fits.gz
1   kplr005780885-2009166043257_lpd-targ.fits.gz
2   kplr005780885-2009291181958_spd-targ.fits.gz
3   kplr005780885-2010326094124_spd-targ.fits.gz
4   kplr005780885-2011024051157_spd-targ.fits.gz
5   kplr005780885-2011053090032_spd-targ.fits.gz
6   kplr005780885-2011073133259_spd-targ.fits.gz
7   kplr005780885-2011177032512_lpd-targ.fits.gz
8   kplr008191672-2010140023957_spd-targ.fits.gz
9   kplr009651668-2009166043257_lpd-targ.fits.gz
10  kplr009651668-2011271113734_lpd-targ.fits.gz
11  kplr009651668-2013131215648_lpd-targ.fits.gz
12  kplr010619192-2009166043257_lpd-targ.fits.gz
13  kplr010874614-2009131105131_lpd-targ.fits.gz
14  kplr010874614-2010140023957_spd-targ.fits.gz
15  kplr010874614-2012088054726_lpd-targ.fits.gz
16  kplr010874614-2013131215648_lpd-targ.fits.gz
17  kplr011446443-2009131105131_lpd-targ.fits.gz
18  kplr011446443-2009131110544_spd-targ.fits.gz
19  kplr

In [2]:
dc.shape

(35, 1)

In [3]:
df.shape

(35, 1)